## 00 — What Are Vector Tiles?

In Module 06 we identified four pain points in our handbuilt system:
1. Slow startup (load + index all data upfront)
2. Verbose GeoJSON format
3. Whole file always resident in memory
4. No streaming — unused regions still loaded

**Vector tiles** are the data format that solves all four. This notebook explains what they are before we use the tool that generates them.

## The Core Idea — Pre-Sliced, Pre-Indexed Data

Instead of four large files that cover the whole world, a vector tile pyramid pre-slices the world into thousands of small tiles — one per `{zoom}/{x}/{y}` address — before the user ever opens the map.

```
Zoom 0: 1 tile  (whole world)
Zoom 1: 4 tiles (quadrants)
Zoom 2: 16 tiles
...
Zoom 14: 268 million tiles (most empty)
```

When the user views a map, only the tiles that are currently visible are fetched. A user in Paris at zoom 12 receives ~12 tiles covering roughly 5km × 5km each. Siberia is never touched.

## How Each Tile Maps to Our System

Every choice we made manually now happens automatically inside the tile generator:

| What we built | What the tile system does |
|---------------|---------------------------|
| 4 LOD files at fixed epsilons | Per-zoom simplification baked into each tile |
| Grid index bucketing features into cells | Each tile IS a cell — features are pre-bucketed by definition |
| Viewport bbox culling | Each tile covers a fixed bbox — fetching only nearby tiles IS the cull |
| Zoom decision function | The tile URL scheme `/{z}/{x}/{y}` carries the zoom level |
| GeoJSON text format | MVT binary encoding — coordinates as integers, ~5× smaller |
| Whole file loaded at startup | Each tile fetched on demand, ~50–200 KB each |

## The Tile Coordinate System

Tiles use `(z, x, y)` addressing. At zoom `z`, the world is divided into a `2^z × 2^z` grid.

Given a longitude/latitude, we can compute its tile address:

In [4]:
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y

# Paris
for zoom in [2, 5, 8, 12]:
    z, x, y = lon_lat_to_tile(2.35, 48.86, zoom)
    print(f"  zoom {z:>2}  tile ({z}/{x}/{y})")

  zoom  2  tile (2/2/1)
  zoom  5  tile (5/16/11)
  zoom  8  tile (8/129/88)
  zoom 12  tile (12/2074/1409)


## The MVT Binary Format

Mapbox Vector Tiles (MVT) store geometry as integers instead of floating-point text.

Each tile has a local coordinate space of 4096 × 4096 units. A coordinate like `(48.8566, 2.3522)` is projected into this space and stored as two small integers (e.g., `(2047, 1803)`).

This gives:
- **5–10× smaller files** vs. GeoJSON (integers compress better than decimal strings)
- **Faster parse** (no string-to-float conversion)
- **Lossy but controlled precision** (4096 units per tile at zoom 14 ≈ 2m resolution)

## The PMTiles Format

Traditionally, tile pyramids were stored in SQLite databases (`.mbtiles`) or as millions of individual files on a server.

**PMTiles** is a newer single-file format that stores the entire tile pyramid in one `.pmtiles` file, arranged so that spatially nearby tiles are stored close together on disk. A client can fetch just the tiles it needs using HTTP range requests — no tile server required, just a static file on any CDN.

For our purposes: `tippecanoe` can output either `.mbtiles` or `.pmtiles`.

## Exercise A

At zoom 12, the world is divided into `2^12 × 2^12 = 4096 × 4096 = ~16.7 million` tiles.

1. How many tiles cover Western Europe at zoom 12? (Approximate using the bounding box [-10, 35, 30, 60])
2. If each tile is 100 KB on average, how much data would the user need to download to view all of Western Europe at zoom 12?

Compare that to loading our `extra_fine` GeoJSON for the same region.

In [5]:
# Calculate tile count for Western Europe at zoom 12
# Estimate download size vs. GeoJSON approach
from pathlib import Path
import math


def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y


def tile_count_for_bbox(bbox, zoom):
    """Count how many Web Mercator tiles intersect a lon/lat bbox."""
    min_lon, min_lat, max_lon, max_lat = bbox

    # x increases west to east. y increases north to south.
    _, x_left,  y_top    = lon_lat_to_tile(min_lon, max_lat, zoom)
    _, x_right, y_bottom = lon_lat_to_tile(max_lon, min_lat, zoom)

    x_min, x_max = sorted([x_left, x_right])
    y_min, y_max = sorted([y_top, y_bottom])

    x_tiles = x_max - x_min + 1
    y_tiles = y_max - y_min + 1
    total   = x_tiles * y_tiles

    return {
        "x_range": (x_min, x_max),
        "y_range": (y_min, y_max),
        "x_tiles": x_tiles,
        "y_tiles": y_tiles,
        "total_tiles": total,
    }


western_europe_bbox = [-10, 35, 30, 60]  # [min_lon, min_lat, max_lon, max_lat]
zoom = 12
info = tile_count_for_bbox(western_europe_bbox, zoom)

avg_tile_kb = 100
download_mb = info["total_tiles"] * avg_tile_kb / 1000

extra_fine = Path("../../data/lod/railroads_extra_fine.geojson")
print(f"Western Europe bbox: {western_europe_bbox}")
print(f"Zoom level: {zoom}")
print(f"Tile x range: {info['x_range']}")
print(f"Tile y range: {info['y_range']}")
print(f"Tiles across x: {info['x_tiles']:,}")
print(f"Tiles across y: {info['y_tiles']:,}")
print(f"Total tiles: {info['total_tiles']:,}")
print(f"Estimated download at {avg_tile_kb} KB/tile: {download_mb:,.1f} MB")

if extra_fine.exists():
    geojson_mb = extra_fine.stat().st_size / 1_000_000
    print(f"Extra-fine GeoJSON file size: {geojson_mb:.1f} MB")
    print(f"Tile estimate / extra-fine GeoJSON: {download_mb / geojson_mb:.1f}×")
else:
    print("Extra-fine GeoJSON not found locally, but the key comparison is this:")
    print("a tile system downloads only the visible non-empty tiles, while GeoJSON loads the whole file first.")


Western Europe bbox: [-10, 35, 30, 60]
Zoom level: 12
Tile x range: (1934, 2389)
Tile y range: (1189, 1622)
Tiles across x: 456
Tiles across y: 434
Total tiles: 197,904
Estimated download at 100 KB/tile: 19,790.4 MB
Extra-fine GeoJSON not found locally, but the key comparison is this:
a tile system downloads only the visible non-empty tiles, while GeoJSON loads the whole file first.


## Exercise B

The tile coordinate formula uses the Web Mercator projection — the same projection used by Google Maps, OpenStreetMap, and virtually all web maps.

Web Mercator distorts areas significantly near the poles. Greenland appears roughly the same size as Africa on a Web Mercator map, even though Africa is ~14× larger.

Does this distortion affect the **accuracy** of our railroad visualization? Explain why or why not in 3–4 sentences.

In [6]:
# Web Mercator distortion does not make the railroad visualization wrong by itself.
# The railroad coordinates are still placed consistently on the same Web Mercator basemap,
# so the lines appear in the correct relative locations for web-map viewing.
# The distortion would matter if we used this view to measure true areas, true distances,
# or compare the real size of regions near the poles against regions near the equator.
# For visualizing where railroad lines are and letting the tile system draw them on a map,
# the projection is acceptable because the data and the basemap use the same projection.


## Check Your Understanding

The tile grid at zoom 14 has ~268 million possible tile addresses. Most tiles — over oceans, deserts, and polar regions — contain no data.

Both `.mbtiles` (SQLite) and `.pmtiles` (single file) only store non-empty tiles. Why is this critical, and how does it relate to the `scalerank` filtering decision we made in our LOD pipeline?

---

**Check Your Understanding Answer**

Storing only non-empty tiles is critical because the full tile address space is enormous. At zoom 14 there are hundreds of millions of possible tile addresses, but most of those places do not contain railroad data, so writing empty tiles would waste storage and slow down tile lookup. This is the same basic idea as our `scalerank` decision: do not carry low-value data into places or zoom levels where the user will not benefit from seeing it. The difference is that vector tiles make the storage sparse automatically by tile, while `scalerank <= 4` was a manual feature-quality filter that reduced the data before writing our simplified GeoJSON files.

## Next

In [01 — Using Tippecanoe](./01-Using_Tippecanoe.ipynb), we run `tippecanoe` on the raw railroad GeoJSON and map each of its flags to decisions we already made.